# ManyFEWS — how the flood forecast works

This notebook walks through the whole ManyFEWS forecasting chain one stage at a
time, plotting what comes out of each. It runs entirely in Colab: no database, no
message broker, no API keys.

    weather forecast  →  evapotranspiration  →  soil store  →  routing
                      →  river flow  →  flood depth  →  map

The study catchment is **Majalaya**, West Java, Indonesia — 212 km² of the upper
Citarum, upstream of a town that floods regularly.

**Two things worth knowing before you start.**

1. An ordinary forecast for this catchment floods *nothing*. Typical flows are
   10–30 m³/s and the flood model does not register anything until 50 m³/s. The
   empty map you will see partway through is the correct answer, not a bug.
2. To see the flood model do anything you therefore have to force it — either by
   injecting a synthetic storm, or by driving the inundation model directly from
   a chosen river flow. Both are near the end.

Runtime: about two minutes, most of it the initial download.

## Setup

In [ ]:
# Put the core package on the path, cloning the repository (~25 MB of model
# data) if we are not already inside a checkout.
import subprocess, sys
from pathlib import Path

REPO = "https://github.com/simreaney/ManyFEWS.git"
BRANCH = "main"          # set this if core/ lives on a different branch

def find_core():
    """Look for core/manyfews_core here, in any parent, or in a clone."""
    for base in (Path.cwd(), *Path.cwd().parents, Path("ManyFEWS")):
        candidate = base / "core"
        if (candidate / "manyfews_core" / "__init__.py").is_file():
            return candidate
    return None

core = find_core()
if core is None:
    if not Path("ManyFEWS").is_dir():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO], check=True
        )
    core = find_core()

if core is None:
    raise SystemExit(
        f"Could not find the manyfews_core package.\n\n"
        f"The clone of {REPO} (branch {BRANCH}) has no core/ directory, so there "
        f"is nothing to import.\n"
        f"That directory has to be committed and pushed before this notebook can "
        f"run in Colab.\n"
        f"If it lives on another branch, set BRANCH above and re-run this cell."
    )

sys.path.insert(0, str(core))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "folium"], check=True)

import manyfews_core as mf
from manyfews_core.plotting import apply_style
apply_style()

print(f"manyfews_core {mf.__version__}")
print(f"data directory: {mf.data_dir()}")
for name in ("RainfallRunoffModelParameters.csv",
             "floodEmulatorParams-20230921.csv",
             "channel.geojson"):
    size = mf.data_path(name).stat().st_size / 1e6
    print(f"  {name:<38} {size:7.2f} MB")

## Parameters

Everything configurable lives here. Edit a value, then re-run this cell and the
ones below it.

The catchment means (`latitude_deg`, `altitude_m`, `area_km2`) drive the
hydrology. They are deliberately separate from `weather_lat`/`weather_lon`, which
is only the point where the forecast is sampled — in the original Django code
these three constants were hardcoded inside a function body, which is why it only
ever worked for one catchment.

In [ ]:
CATCHMENT = mf.CatchmentConfig(
    name="Majalaya",
    latitude_deg=-7.125,     # catchment mean, for solar geometry
    altitude_m=1157.0,       # catchment mean, for atmospheric pressure
    area_km2=212.2640,
    weather_lat=-7.05,       # where the forecast is sampled
    weather_lon=107.758,
)

FORECAST = mf.ForecastConfig(
    model="gfs_seamless",
    forecast_days=16,
    max_members=10,          # 0 or None keeps every member Open-Meteo offers
    spinup_days=29,
)

print(CATCHMENT)
print(FORECAST)

## Stage 1 — Weather

Open-Meteo serves both an ensemble forecast and a reanalysis archive, free and
without an API key. The model wants six-hourly buckets, so hourly values are
aggregated: rainfall sums, temperature keeps the min and max within each bucket,
humidity and wind average.

Wind arrives as speed and direction and is decomposed into components, using the
meteorological convention that direction is the one the wind blows *from*.

In [ ]:
forecast = mf.fetch_forecast(CATCHMENT, FORECAST)

print(f"{len(forecast)} ensemble members, {len(forecast[0])} six-hour buckets each")
print(f"covering {forecast[0].times[0]} to {forecast[0].times[-1]}\n")
for member in forecast[:3]:
    print(" ", member.summary())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from manyfews_core.plotting import RAIN, BAND_INNER, LINE_MEDIAN, INK_SECONDARY
from manyfews_core.weather import WeatherSeries

control = forecast[0]
t = control.times.astype("datetime64[m]").tolist()

fig, axes = plt.subplots(3, 1, figsize=(9.5, 6.5), sharex=True,
                         gridspec_kw={"hspace": 0.12})

axes[0].bar(t, control.data[:, WeatherSeries.PRECIP], width=0.22, color=RAIN, lw=0)
axes[0].set_ylabel("Rain (mm / 6 h)")
axes[0].set_title("Control member — the raw ingredients")

axes[1].fill_between(t, control.data[:, WeatherSeries.TMIN] - 273.15,
                     control.data[:, WeatherSeries.TMAX] - 273.15,
                     color=BAND_INNER, alpha=0.35, lw=0, label="min–max")
axes[1].set_ylabel("Temperature (°C)")
axes[1].legend(loc="upper right", fontsize=9, labelcolor=INK_SECONDARY)

axes[2].plot(t, control.data[:, WeatherSeries.RH], color=LINE_MEDIAN, lw=2)
axes[2].set_ylabel("Humidity (%)")

fig.autofmt_xdate()
plt.show()

## Stage 2 — Evapotranspiration

FAO56 Penman-Monteith turns temperature, humidity, wind and solar geometry into
how much water the vegetation and soil give back to the atmosphere. Solar
radiation is not measured — it is estimated from the diurnal temperature range
via the Hargreaves relation, which is why the day/night swing matters so much
here.

The same function also returns open-water evaporation as a by-product. The
original pipeline computes it and then throws it away.

In [ ]:
from manyfews_core.hydrology import FAO56

n = len(control)
temp_max = control.data[:, WeatherSeries.TMAX] - 273.15
temp_min = control.data[:, WeatherSeries.TMIN] - 273.15

eto, e0 = FAO56(
    dt=CATCHMENT.timestep_days,
    predictionDate=control.start,
    Tmin=np.repeat(temp_min.reshape(n // 4, 4).min(axis=1), 4),
    Tmax=np.repeat(temp_max.reshape(n // 4, 4).max(axis=1), 4),
    alt=CATCHMENT.altitude_m,
    lat=CATCHMENT.latitude_deg,
    T=(temp_min + temp_max) / 2,
    u2=np.full(n, 1.2),
    RH=control.data[:, WeatherSeries.RH],
)

fig, ax = plt.subplots(figsize=(9.5, 3.6))
ax.plot(t, e0, color=BAND_INNER, lw=2, label="Open water evaporation")
ax.plot(t, eto, color=LINE_MEDIAN, lw=2, label="Reference evapotranspiration")
ax.set_ylabel("mm / day")
ax.set_title("Atmospheric demand for water")
ax.legend(loc="upper right", fontsize=9, labelcolor=INK_SECONDARY)
fig.autofmt_xdate()
plt.show()

print(f"mean reference ET: {eto.mean():.2f} mm/day")

## Stage 3 — The soil store

The Probability Distributed Model holds a single bucket of catchment storage. As
it fills, a growing fraction of incoming rain runs straight off instead of
soaking in; water already stored drains away slowly.

The catchment is not calibrated to one set of parameters but to **100** of them,
each a plausible fit to the observed record. That spread is the model's own
admission of uncertainty, and it is carried all the way to the final map. Below
is a single one of the hundred.

In [ ]:
from manyfews_core.hydrology import PDMmodel

params = mf.load_parameters()
print(f"{params.shape[0]} parameter sets — Smax, qmax, k, Tr")

smax, qmax, k, tr = params[0]
qp = control.data[:, WeatherSeries.PRECIP] / CATCHMENT.timestep_days
qro, qd, ea, storage = PDMmodel(qp, eto, smax, 1, k, CATCHMENT.timestep_days, smax / 2)

fig, axes = plt.subplots(2, 1, figsize=(9.5, 5.4), sharex=True,
                         gridspec_kw={"hspace": 0.12})

axes[0].fill_between(t, storage, color=BAND_INNER, alpha=0.5, lw=0)
axes[0].axhline(smax, color="#d03b3b", lw=1.5, ls=(0, (5, 3)))
axes[0].annotate(f"capacity — {smax:.0f} mm", xy=(0.995, smax), xycoords=("axes fraction", "data"),
                 xytext=(0, 4), textcoords="offset points", ha="right",
                 fontsize=9, color="#d03b3b", weight="600")
axes[0].set_ylabel("Storage (mm)")
axes[0].set_title(f"Soil store, parameter set 1 of {params.shape[0]}")

axes[1].plot(t, qro, color=LINE_MEDIAN, lw=2, label="Surface runoff (fast)")
axes[1].plot(t, qd, color=BAND_INNER, lw=2, label="Drainage (slow)")
axes[1].set_ylabel("mm / day")
axes[1].legend(loc="upper right", fontsize=9, labelcolor=INK_SECONDARY)

fig.autofmt_xdate()
plt.show()

## Stage 4 — Routing

Runoff does not reach the river instantly. Two stores delay it: a linear one for
slow drainage, and a non-linear one (`q = a·v^{5/3}`) for the fast surface
response. Their sum, converted from mm/day over the catchment to m³/s, is the
river flow.

In [ ]:
from manyfews_core.hydrology import RoutingFun

dt = CATCHMENT.timestep_days
slow = RoutingFun(qd, tr, 1, dt, 2.0)
fast = RoutingFun(qro, qmax, 5 / 3, dt, 2.0)
to_cumecs = CATCHMENT.area_km2 * 1e3 / 24 / 3600

fig, ax = plt.subplots(figsize=(9.5, 4))
ax.stackplot(t, slow * to_cumecs, fast * to_cumecs,
             colors=["#86b6ef", "#2a78d6"], labels=["Slow (baseflow)", "Fast (storm response)"])
ax.set_ylabel("River flow (m³/s)")
ax.set_title("Where the water in the river comes from")
ax.legend(loc="upper right", fontsize=9, labelcolor=INK_SECONDARY)
fig.autofmt_xdate()
plt.show()

## Stage 5 — The full ensemble

Now all of it at once: every weather member run through every parameter set. With
10 members and 100 parameter sets, each forecast time carries 1,000 samples of
what the river might do.

Before the forecast can run, the model needs to know how wet the catchment
already is. That comes from replaying roughly a month of observed weather from the
archive — the spin-up.

In [ ]:
import time

start = time.time()
history = mf.fetch_history(CATCHMENT, FORECAST)
state = mf.spin_up(history, params, CATCHMENT)
ensemble = mf.run_ensemble(forecast, state, params, CATCHMENT)

print(f"spin-up over {len(history) // 4} days of observed weather")
print(f"ensemble: {ensemble.flow_m3s.shape} (members × steps × parameter sets)")
print(f"took {time.time() - start:.1f}s")

In [ ]:
from manyfews_core.plotting import flow_fan

ax = flow_fan(ensemble.times, ensemble.flow_m3s, show_traces=True,
              title="River flow forecast — every member, every parameter set")
plt.show()

## Stage 6 — The flood emulator

Running a 2D hydraulic model for every forecast, every six hours, is not
feasible. So it was run offline across a range of flows, and a cubic was fitted
per ground cell:

$$\text{depth}(Q) = \max\bigl(0,\; P_0 + P_1 Q + P_2 Q^2 + P_3 Q^3\bigr) \quad\text{if } Q \ge Q_{\min}$$

302,748 cells at roughly 2 m resolution, each with its own four coefficients and
its own threshold. There is no spatial coupling and no memory: **the entire flood
map is a function of one number.**

That is also the model's weakness. Outside the flow range it was fitted over, a
cubic does what cubics do.

In [ ]:
emulator = mf.FloodEmulator.from_csv()
print(f"{emulator.n_cells:,} cells, {emulator.cell_size * 111_000:.1f} m each")
print(f"flood thresholds range {emulator.min_q.min():.0f}–{emulator.min_q.max():.0f} m³/s")

for q in (400, 500, 800):
    raw = emulator.beta0 + q * (emulator.beta1 + q * (emulator.beta2 + q * emulator.beta3))
    print(f"  unclamped max depth at Q={q}: {raw.max():8.1f} m")
print(f"\n  clamped at Q={emulator.q_cap:.0f}: {emulator.depth_at([1e6]).max():.2f} m")

In [ ]:
from manyfews_core.plotting import depth_vs_flow

depth_vs_flow(emulator, n_cells=200, q_max=600)
plt.show()

In [ ]:
# Wet-cell count against flow. Note it *falls* past the calibration limit, as
# cubics turn over and go negative — which is what the clamp exists to prevent.
flows_axis = np.arange(0, 420, 10, dtype=float)
raw_wet = []
for q in flows_axis:
    raw = emulator.beta0 + q * (emulator.beta1 + q * (emulator.beta2 + q * emulator.beta3))
    raw = np.where(q < emulator.min_q, 0.0, np.maximum(raw, 0.0))
    raw_wet.append((raw > 0.01).sum())

fig, ax = plt.subplots(figsize=(8.5, 3.8))
ax.plot(flows_axis, np.array(raw_wet) / 1000, color=LINE_MEDIAN, lw=2)
ax.axvline(emulator.q_cap, color="#d03b3b", lw=1.5, ls=(0, (5, 3)))
ax.annotate("calibration limit", xy=(emulator.q_cap, ax.get_ylim()[1] * 0.5),
            xytext=(-8, 0), textcoords="offset points", ha="right",
            fontsize=9, color="#d03b3b", weight="600")
ax.set_xlabel("River flow (m³/s)")
ax.set_ylabel("Flooded cells (thousands)")
ax.set_title("Unclamped flood extent against flow")
plt.show()

## Reality check — the real forecast

Now put the two halves together and ask what today's actual forecast does to the
flood model.

In [ ]:
peak = ensemble.peak_step()
pooled = ensemble.pooled(peak)

print(f"peak forecast step: {ensemble.times[peak]}")
print(f"pooled flow across {pooled.size} samples:")
for pct in (10, 50, 90, 100):
    print(f"    p{pct:<3} {np.percentile(pooled, pct):7.1f} m³/s")
print(f"\nlowest flood threshold anywhere: {emulator.min_q.min():.0f} m³/s")

field = emulator.field(pooled)
wet = field.wet_cells(90)
if wet == 0:
    print("\n→ No flooding forecast. Every cell is dry at every percentile.")
    print("  This is the normal outcome — the catchment floods rarely.")
else:
    print(f"\n→ {wet:,} cells flooded at the 90th percentile.")

In [ ]:
ax = flow_fan(ensemble.times, ensemble.flow_m3s,
              title="Today's forecast against the flooding threshold")
ax.set_ylim(0, max(60, np.percentile(pooled, 99) * 1.2))
plt.show()

## Scenario A — a synthetic storm

To see the system respond, force it. This replaces one day's forecast rainfall
with a design storm and re-runs everything downstream.

The Django application's test mode uses 100 mm. As the sweep below shows, that
is not enough to flood anything here.

In [ ]:
STORM_TOTAL_MM = 200.0     # try 100 to reproduce the Django test-mode default
STORM_DAYS_AHEAD = 2

storm = mf.StormConfig(enabled=True, total_mm=STORM_TOTAL_MM, days_ahead=STORM_DAYS_AHEAD)
stormy = mf.inject_storm_ensemble(forecast, forecast[0].start, storm)
storm_ensemble = mf.run_ensemble(stormy, state, params, CATCHMENT)

storm_peak = storm_ensemble.peak_step()
storm_pooled = storm_ensemble.pooled(storm_peak)
storm_field = emulator.field(storm_pooled)

print(f"peak flow p50 {np.percentile(storm_pooled, 50):.1f}  "
      f"p90 {np.percentile(storm_pooled, 90):.1f} m³/s")
print(f"flooded cells: {storm_field.wet_cells(50):,} at p50, "
      f"{storm_field.wet_cells(90):,} at p90")
print(f"deepest: {storm_field.max_depth(90):.2f} m")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9.5, 7), sharex=True,
                         gridspec_kw={"hspace": 0.18})
flow_fan(ensemble.times, ensemble.flow_m3s, ax=axes[0], title="Unmodified forecast")
flow_fan(storm_ensemble.times, storm_ensemble.flow_m3s, ax=axes[1],
         title=f"With a {STORM_TOTAL_MM:.0f} mm storm on day {STORM_DAYS_AHEAD}")
axes[1].set_ylim(axes[0].get_ylim()[0], max(axes[1].get_ylim()[1], 60))
plt.show()

In [ ]:
# How big does the storm have to be? This is the chart that shows why 100 mm
# is not a usable test default.
from manyfews_core.plotting import storm_response_curve

totals = [50, 100, 150, 200, 300]
p50s, p90s = [], []
for mm in totals:
    cfg = mf.StormConfig(enabled=True, total_mm=mm, days_ahead=STORM_DAYS_AHEAD)
    ens_mm = mf.run_ensemble(mf.inject_storm_ensemble(forecast, forecast[0].start, cfg),
                             state, params, CATCHMENT)
    sample = ens_mm.pooled(ens_mm.peak_step())
    p50s.append(np.percentile(sample, 50))
    p90s.append(np.percentile(sample, 90))
    print(f"  {mm:3d} mm → p50 {p50s[-1]:6.1f}  p90 {p90s[-1]:6.1f} m³/s")

storm_response_curve(totals, p50s, p90s)
plt.show()

## Scenario B — drive the flood model directly

Because depth depends only on flow, you can skip the hydrology entirely and ask
the inundation model what a given river flow looks like on the ground. This is
instant, and it is the honest way to explore the emulator on its own terms.

Cells inside the river channel are masked out — the emulator reports metres of
water there, but that is just the river.

In [ ]:
FLOW_M3S = 160.0        # anything from 50 (nothing floods) to 300 (calibration limit)

channel = mf.cached_channel_mask(emulator)
direct = emulator.field(mf.constant_flow_samples(FLOW_M3S), channel_mask=channel)

print(f"at {FLOW_M3S:.0f} m³/s:")
print(f"  flooded cells   {direct.wet_cells(50):,}")
print(f"  deepest         {direct.max_depth(50):.2f} m")
print(f"  mean where wet  {direct.mean_wet_depth(50):.2f} m")
print(f"  channel masked  {channel.sum():,} cells")

In [ ]:
from manyfews_core.plotting import depth_histogram

depth_histogram(direct, pct=50)
plt.show()

In [ ]:
from manyfews_core.mapping import flood_map

raster = mf.rasterise(emulator, direct.layer(50), mask=channel)
flood_map(raster, vmax=3.0)

## Optional — an interactive slider

Recomputing the whole 302,748-cell depth field takes about 0.2 seconds, so a
slider is perfectly viable. Widgets in Colab need the custom widget manager,
which does not always survive being shared, so this cell degrades gracefully: if
it does not work, edit `FLOW_M3S` above and re-run instead.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
    import ipywidgets as widgets
    from IPython.display import display, clear_output

    slider = widgets.FloatSlider(value=160.0, min=50.0, max=300.0, step=10.0,
                                 description="Flow (m³/s)", continuous_update=False,
                                 style={"description_width": "initial"},
                                 layout=widgets.Layout(width="520px"))
    out = widgets.Output()

    def redraw(change):
        with out:
            clear_output(wait=True)
            f = emulator.field(mf.constant_flow_samples(change["new"]), channel_mask=channel)
            print(f"{f.wet_cells(50):,} cells flooded, deepest {f.max_depth(50):.2f} m")
            display(flood_map(mf.rasterise(emulator, f.layer(50), mask=channel)))

    slider.observe(redraw, names="value")
    display(slider, out)
    redraw({"new": slider.value})
except Exception as exc:
    print(f"Widgets unavailable ({type(exc).__name__}: {exc}).")
    print("Edit FLOW_M3S in the cell above and re-run it instead.")

## Caveats

**The depth model is a statistical emulator, not hydraulics.** Each cell's cubic
was fitted to outputs of a 2D hydraulic model. It reproduces that model's answers
within the range it was fitted over and is meaningless outside it — which is why
the input is clamped at 300 m³/s here. The original Django application applies no
such clamp and will happily report a 118 m flood.

**Depth depends on flow alone.** No hydrograph shape, no antecedent wetness in
the floodplain, no breach or blockage, no timing. Two very different storms that
produce the same peak flow produce identical maps.

**Uncertainty here is parametric, not structural.** The spread comes from 100
rainfall-runoff parameter sets and the weather ensemble. It says nothing about
whether the model structure is right, or whether the hydraulic model it emulates
was well calibrated.

**The channel mask is approximate.** Cells are masked when their centre falls
inside the mapped channel; a cell the river merely clips is kept.

**This is a demonstration, not an operational warning system.** The live service
adds scheduling, alerting and human oversight — deliberately not ported here.